# PKG counterparty threshold exploration — `pkg_cpty_threshold_explore.ipynb`

**Question.** Which counterparties should get the full counterparty metric set? Most counterparty ids are seen once,
for a small amount. Before building any counterparty metric, this notebook measures how many there are, how much money
they carry, and what each candidate threshold on **transactions (k)** and/or **amount ($)** would keep and drop.

**Unit.** The entity `ent_l2` from `pkg_cpty_entity_map` (v1) is the primary unit; raw `cpty_id` is shown for comparison
(§9). A threshold has to be applied to the entity, not the raw id: CIGNA alone is ~258K ids, most of them single small
payments. Map v1 still merges some bare first names (JOHN, DAVID); that touches a handful of entities and does not
move these distributions.

**Window.** Trailing `WINDOW_MONTHS` months ending `LAST_MONTH`. Eligibility for counterparty metrics will be decided on a
trailing window, so the threshold is studied on one; §8 compares with a single month.

**Direction (counterparty view).** `in` = the counterparty receives from PNC customers (B categories);
`out` = the counterparty pays PNC customers (C categories). Everything is the PNC-visible slice only.

**Inputs** (all in `bdahd01p_dlcdi1_cdi_tm`): `pkg_edge_monthly`, `pkg_cpty_entity_map`, `pkg_cpty_dim`, `pkg_cust_dim`.
No Hive table is created. Intermediates are written as parquet under `<cdi_tm location>/_tmp_pkg_cpty_explore/<window>/`,
reused on reruns (`FORCE` rebuilds), and deleted in §12 when you are done. Charts go to `OUT_DIR`; tables stay inline.

| § | Content |
|---|---|
| 0 | Setup, config, helpers |
| 1 | Input checks |
| 2 | Build: window pairs → raw-id profile → unit profiles |
| 3 | Headline numbers per unit |
| 4 | Distributions: transactions, amount, PNC customers, active months |
| 5 | Joint distribution and the single-transaction tail |
| 6 | Threshold trade-off: candidate rules, k × $ grids, rule search under floors |
| 7 | Chosen rule: what gets dropped, with examples |
| 8 | 12-month window vs a single month |
| 9 | Raw id vs entity |
| 10 | Customer-side coverage |
| 11 | Save outputs |
| 12 | Cleanup |

## §0 Setup

In [ ]:
# §0a  Session and compat shims
import os, json, time, warnings, html as _html
warnings.filterwarnings("ignore", category=ResourceWarning)       # py4j socket noise
warnings.filterwarnings("ignore", category=DeprecationWarning)     # distutils in Cloudera PySpark 3.3
import numpy as np
import pandas as pd

# Cloudera SPARK3 PySpark 3.3 vs NumPy 2 / pandas 2 in conda env new_default
for _alias, _target in [("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                        ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

import matplotlib.pyplot as plt
from IPython.display import display, HTML
from pyspark.sql import SparkSession, functions as F

SHUFFLE_PARTITIONS = 2000      # pair table is several hundred million rows; AQE coalesces small ones

spark = (SparkSession.builder
         .appName("pkg_cpty_threshold_explore")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.executor.memoryOverhead", "2g")
         .config("spark.dynamicAllocation.enabled", "true")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true")
         .enableHiveSupport()
         .getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
# §0b  Config
DB         = "bdahd01p_dlcdi1_cdi_tm"
T_EDGE_M   = f"{DB}.pkg_edge_monthly"
T_CPTY_DIM = f"{DB}.pkg_cpty_dim"
T_MAP      = f"{DB}.pkg_cpty_entity_map"      # pkg_cpty_entity_map_v2 → name_class from the map (typer v2)
T_CUST_DIM = f"{DB}.pkg_cust_dim"

HISTORY_START = "2024-01"      # first month in the edge tables
LAST_MONTH    = "2026-08"      # last complete month
WINDOW_MONTHS = 12             # eligibility window; 3 gives a quick timing run
LEVELS        = ["ent_l2", "cpty_id"]   # first = primary unit; add "ent_l1" to compare
FORCE         = False          # rebuild cached intermediates even if present
RUN_CUST_COVERAGE = True       # §10: one more join over the pair table
MAX_MAP_MISS  = 0.001          # share of window pairs whose cpty_id is missing from the entity map
MAX_DIM_MISS  = 0.001          # share of window ids missing from pkg_cpty_dim

# Threshold grids. Any rule whose thresholds sit on these grids is evaluated exactly from the §3 cube.
K_GRID = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25, 50, 100, 1000]                 # transactions in window
A_GRID = [0, 10, 25, 50, 100, 250, 500, 1_000, 2_500, 5_000, 10_000, 25_000,
          50_000, 100_000, 250_000, 1_000_000, 10_000_000]                           # dollars in window
D_GRID = [1, 2, 3, 4, 5, 10, 25, 100, 1000]                                         # distinct PNC customers
M_GRID = [m for m in [1, 2, 3, 4, 6, 9, 12, 18, 24] if m <= WINDOW_MONTHS]         # active months in window

# Candidate rules. Keys: n = transactions, amt = dollars, cust = distinct PNC customers, months = active months.
# Conditions combine with `logic` ("or" by default, or "and"). {} keeps everything.
# Optional "classes": the rule keeps only units whose name_class is listed (others are dropped).
BIZ_LIKE = ["business", "unknown", "bank", "numeric_name"]     # v2 classes that are not individuals
CANDIDATE_RULES = [
    {},
    {"n": 2},
    {"n": 3},
    {"amt": 1_000},
    {"amt": 10_000},
    {"n": 2, "amt": 500, "logic": "and"},
    {"n": 2, "amt": 1_000},
    {"n": 3, "amt": 5_000},
    {"n": 3, "amt": 10_000},
    {"n": 5, "amt": 10_000},
    {"cust": 2, "amt": 10_000},
    {"months": 2, "amt": 10_000},
    {"amt": 1_000, "classes": BIZ_LIKE},
    {"amt": 5_000, "classes": BIZ_LIKE},
    {"amt": 10_000, "classes": BIZ_LIKE},
]
CHOSEN_RULE = {"n": 3, "amt": 5_000}   # drives §7 and is highlighted elsewhere; revisit after §6

# Floors for the automatic rule search in §6: share of the window total a rule must keep
KEEP_FLOORS = {"amt": 0.99, "amt_tm": 0.99}

# ---- derived ----
def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"

WIN_MONTHS = [month_add(LAST_MONTH, -i) for i in range(WINDOW_MONTHS - 1, -1, -1)]
WIN_START  = WIN_MONTHS[0]
WIN_BASE   = int(WIN_START[:4]) * 12 + int(WIN_START[5:7])   # month index = offset from window start
assert WIN_START >= HISTORY_START, f"window starts {WIN_START}, before {HISTORY_START}"
assert 1 <= WINDOW_MONTHS <= 62, "month presence mask is a bigint"

_ALLOWED = {"n", "amt", "cust", "months", "logic", "classes"}
for _r in CANDIDATE_RULES + [CHOSEN_RULE]:
    assert set(_r) <= _ALLOWED and _r.get("logic", "or") in ("or", "and"), f"bad rule {_r}"
if CHOSEN_RULE not in CANDIDATE_RULES:
    CANDIDATE_RULES.append(CHOSEN_RULE)

MAP_TAG = "" if T_MAP.endswith(".pkg_cpty_entity_map") else "_" + T_MAP.split(".")[-1].replace("pkg_cpty_entity_map_", "map")
RUN_TAG = f"w{WIN_START}_{LAST_MONTH}{MAP_TAG}"     # separate cache per map
OUT_DIR = f"../metrics/cpty_explore/{RUN_TAG}"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"window {WIN_START} → {LAST_MONTH} ({WINDOW_MONTHS} months) | levels {LEVELS} | out {OUT_DIR}")

In [ ]:
# §0c  Helpers: labels, rules, display, file system
LEVEL_NAME = {"ent_l2": "entity (ent_l2)", "ent_l1": "ent_l1 (6C→5C folded)", "cpty_id": "raw id"}

def usd(x):
    x = float(x)
    for div, suf in [(1e9, "B"), (1e6, "M"), (1e3, "k")]:
        if abs(x) >= div:
            return f"${x / div:g}{suf}"
    return f"${x:g}"

def rule_label(r):
    if not r:
        return "all"
    parts = []
    if "n" in r:      parts.append(f"txn≥{r['n']}")
    if "cust" in r:   parts.append(f"cust≥{r['cust']}")
    if "months" in r: parts.append(f"months≥{r['months']}")
    if "amt" in r:    parts.append(f"amt≥{usd(r['amt'])}")
    lab = (" OR " if r.get("logic", "or") == "or" else " AND ").join(parts) or "all"
    if "classes" in r:
        lab += " [biz-like]" if list(r["classes"]) == BIZ_LIKE else f" [{'/'.join(r['classes'])}]"
    return lab

CHOSEN_I = CANDIDATE_RULES.index(CHOSEN_RULE)

def cand_name(i):
    # candidate rules carry an id (R0, R1, …) used in every table and on the frontier chart
    return f"R{i}  {rule_label(CANDIDATE_RULES[i])}" + ("  ◀ chosen" if i == CHOSEN_I else "")

def rule_col(r, sfx=""):
    # Spark condition for a rule. sfx="_lm" evaluates it on last-month columns; a months condition has no
    # single-month meaning, so such rules return None there.
    conds = []
    if "n" in r:    conds.append(F.col("n_txn" + sfx) >= r["n"])
    if "amt" in r:  conds.append(F.col("amt" + sfx) >= r["amt"])
    if "cust" in r: conds.append(F.col("n_cust" + sfx) >= r["cust"])
    if "months" in r:
        if sfx:
            return None
        conds.append(F.col("n_months") >= r["months"])
    out = F.lit(True)
    if conds:
        out = conds[0]
        for c in conds[1:]:
            out = (out | c) if r.get("logic", "or") == "or" else (out & c)
    if "classes" in r:
        out = out & F.col("name_class").isin(list(r["classes"]))
    return out

# Grid index = number of grid thresholds the value reaches, so  value ≥ grid[j]  ⇔  index ≥ j + 1
def idx_col(c, grid):
    col = F.col(c)
    return sum([F.when(col >= t, 1).otherwise(0) for t in grid], F.lit(0))

GRID_OF = {"n": ("i_n", K_GRID), "amt": ("i_a", A_GRID), "cust": ("i_d", D_GRID), "months": ("i_m", M_GRID)}

def on_grid(r):
    return all(r[k] in GRID_OF[k][1] for k in GRID_OF if k in r)

def cube_mask(cube, r):
    conds = []
    for k, (col, grid) in GRID_OF.items():
        if k in r:
            assert r[k] in grid, f"{k}={r[k]} is not on the grid {grid}"
            conds.append(cube[col].to_numpy() >= grid.index(r[k]) + 1)
    mask = np.ones(len(cube), bool) if not conds else (
        np.logical_or.reduce(conds) if r.get("logic", "or") == "or" else np.logical_and.reduce(conds))
    if "classes" in r:
        mask = mask & cube["name_class"].isin(list(r["classes"])).to_numpy()
    return mask

def grid_labels(grid, money=False):
    f = usd if money else (lambda v: f"{v:,}")
    labs = {0: f"<{f(grid[0])}"}
    for i in range(1, len(grid) + 1):
        lo = grid[i - 1]
        if i == len(grid):
            labs[i] = f"{f(lo)}+"
        elif money:
            labs[i] = f"{f(lo)}–{f(grid[i])}"
        elif grid[i] - lo == 1:
            labs[i] = f(lo)
        else:
            labs[i] = f"{f(lo)}–{f(grid[i] - 1)}"
    return labs

# ---- display: one output per cell ----
def _fmt_val(v, kind):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if kind == "pct":
        v = 0.0 if abs(v) < 5e-7 else v          # float noise, e.g. -0.000
        return "0" if v == 0 else (f"{v:.3f}" if abs(v) < 1 else f"{v:.2f}")
    if kind == "musd":
        return f"{v:,.1f}"
    if kind in ("usd", "int"):
        return f"{v:,.0f}"
    return f"{v:,.2f}"

def fmt_df(df, fmt=None):
    out = df.copy()
    out.columns.name = None
    for c in out.columns:
        s = out[c]
        if pd.api.types.is_bool_dtype(s):
            continue
        if not pd.api.types.is_numeric_dtype(s):
            out[c] = s.where(s.notna(), "")
            continue
        name = str(c)
        if fmt:
            kind = fmt
        elif "%" in name:
            kind = "pct"
        elif name.startswith("$M"):
            kind = "musd"
        elif name.startswith("$"):
            kind = "usd"
        elif pd.api.types.is_integer_dtype(s) or bool((s.dropna() % 1 == 0).all()):
            kind = "int"
        else:
            kind = "f2"
        out[c] = [_fmt_val(v, kind) for v in s]
    return out

_CSS = ("<style>.pkgt{border-collapse:collapse;font-size:12px;margin-bottom:4px}"
        ".pkgt th,.pkgt td{padding:2px 9px;border-bottom:1px solid #e3e2de;text-align:right;white-space:nowrap}"
        ".pkgt th{background:#e3e2de;color:#0b0b0b !important;font-weight:600}.pkgn{color:#8a8984;margin:2px 0}</style>")

def note(text):
    return ("__note__", text)

def show(*blocks):
    # blocks: "header", note("text"), (title, df) or (title, df, fmt) with fmt in pct/int/usd/musd
    parts = []
    for b in blocks:
        if b is None:
            continue
        if isinstance(b, str):
            parts.append(f"<div style='margin:10px 0 2px;font-size:14px'><b>{_html.escape(b)}</b></div>")
        elif b[0] == "__note__":
            parts.append(f"<div class='pkgn'>{_html.escape(b[1])}</div>")
        else:
            title, df = b[0], b[1]
            fmt = b[2] if len(b) > 2 else None
            parts.append(f"<div style='margin:12px 0 3px'><b>{_html.escape(title)}</b></div>")
            parts.append(fmt_df(df, fmt).to_html(index=False, escape=True, border=0, classes="pkgt"))
    display(HTML(_CSS + "<div style='font-family:\"IBM Plex Sans\",Arial,sans-serif'>" + "".join(parts) + "</div>"))

# ---- charts: saved to disk, never shown inline ----
C_BLUE, C_ORANGE, C_AQUA, C_YELLOW, C_GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e3e2de", "grid.linewidth": 0.6,
                     "axes.edgecolor": "#8a8984", "axes.titlesize": 10, "legend.frameon": False,
                     "lines.linewidth": 2})

def save_fig(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return path

# ---- HDFS helpers (JVM FileSystem; works for hdfs:// and local paths) ----
_jvm, _hconf = spark._jvm, spark._jsc.hadoopConfiguration()

def _fs(p):
    path = _jvm.org.apache.hadoop.fs.Path(p)
    return path.getFileSystem(_hconf), path

def path_done(p):
    fs, path = _fs(p.rstrip("/") + "/_SUCCESS")
    return fs.exists(path)

def path_gb(p):
    fs, path = _fs(p)
    return fs.getContentSummary(path).getLength() / 1e9 if fs.exists(path) else 0.0

def path_delete(p):
    fs, path = _fs(p)
    return bool(fs.exists(path) and fs.delete(path, True))

def db_location(db):
    for r in spark.sql(f"DESCRIBE DATABASE EXTENDED {db}").collect():
        if str(r[0]).strip().lower() == "location":
            return str(r[1]).rstrip("/")
    raise RuntimeError(f"no Location row in DESCRIBE DATABASE {db}")

TMP_ROOT = f"{db_location(DB)}/_tmp_pkg_cpty_explore/{RUN_TAG}"
P_PAIRS  = f"{TMP_ROOT}/pairs"
P_IDS    = f"{TMP_ROOT}/ids"
def P_PROF(level):
    return f"{TMP_ROOT}/profile_{level}"
print("intermediates:", TMP_ROOT)

## §1 Input checks

Fail fast before the expensive pass: required columns, string id types (the 2026-07 int64/str incident), and every
window month present in `pkg_edge_monthly`. Duplicate keys in the map or dims would multiply rows in the joins;
that is caught in §2 by exact reconciliation rather than by an extra pass over 250M rows here.

In [ ]:
# §1  Input checks
def need_cols(table, cols):
    dt = dict(spark.table(table).dtypes)
    miss = [c for c in cols if c not in dt]
    assert not miss, f"{table}: missing {miss}; has {sorted(dt)}"
    return dt

def pick(dt, table, candidates, required=True):
    for c in candidates:
        if c in dt:
            return c
    assert not required, f"{table}: none of {candidates}; has {sorted(dt)}"
    return None

edt = need_cols(T_EDGE_M, ["src_id", "src_kind", "dst_id", "dst_kind", "category", "is_self", "amount", "n_txn", "month"])
mdt = need_cols(T_MAP, ["cpty_id", "ent_l1", "ent_l2"])
MAP_HAS_CLASS = "name_class" in mdt
ddt = need_cols(T_CPTY_DIM, ["cpty_id"])
udt = need_cols(T_CUST_DIM, ["mdm_id", "is_tm"])

# pkg_cpty_dim column names (dominant values per cpty_id)
DIM = {
    "name":           pick(ddt, T_CPTY_DIM, ["cpty_name", "dominant_name", "name"]),
    "cpty_type":      pick(ddt, T_CPTY_DIM, ["cpty_type", "dominant_cpty_type"]),
    "fi":             pick(ddt, T_CPTY_DIM, ["fi", "cpty_fin_entity_name", "dominant_fi"]),
    "is_business":    pick(ddt, T_CPTY_DIM, ["is_business"]),
    "name_is_fi":     pick(ddt, T_CPTY_DIM, ["name_is_fi"]),
    "is_placeholder": pick(ddt, T_CPTY_DIM, ["is_placeholder"]),
    "src":            pick(ddt, T_CPTY_DIM, ["cpty_id_src"], required=False),
}

bad = {}
for t, d, cols in [(T_EDGE_M, edt, ["src_id", "dst_id", "src_kind", "dst_kind", "month"]),
                   (T_MAP, mdt, ["cpty_id", "ent_l1", "ent_l2"]),
                   (T_CPTY_DIM, ddt, ["cpty_id"]),
                   (T_CUST_DIM, udt, ["mdm_id"])]:
    for c in cols:
        if d[c] != "string":
            bad[f"{t}.{c}"] = d[c]
assert not bad, f"id / key columns must be string: {bad}"

parts = set()
for r in spark.sql(f"SHOW PARTITIONS {T_EDGE_M}").collect():
    for kv in r[0].split("/"):
        if kv.startswith("month="):
            parts.add(kv.split("=", 1)[1])
missing = [m for m in WIN_MONTHS if m not in parts]
assert not missing, f"{T_EDGE_M}: window months missing {missing}"

show("§1 Input checks passed",
     ("Columns used", pd.DataFrame([
         {"table": T_EDGE_M, "columns": "src/dst id+kind, category, is_self, amount, n_txn, month"},
         {"table": T_MAP, "columns": "cpty_id, ent_l1, ent_l2" + (", name_class (typer v2)" if MAP_HAS_CLASS else "")},
         {"table": T_CPTY_DIM, "columns": ", ".join(f"{k}={v}" for k, v in DIM.items())},
         {"table": T_CUST_DIM, "columns": "mdm_id, is_tm"}])),
     note(f"Window months present: {WIN_MONTHS[0]} … {WIN_MONTHS[-1]} ({len(WIN_MONTHS)}); "
          f"{T_EDGE_M} holds {len(parts)} months ({min(parts)} → {max(parts)})."))

## §2 Build

Three cached steps, each reconciled exactly:

1. **pairs** — one row per (`cpty_id`, PNC customer, direction) over the window: amount, transactions, month presence
   mask (bit *i* = month *i* from window start), rail mask (bit = category number), self and last-month amounts; with
   `ent_l1`/`ent_l2` from the map and the customer's TM flag. The one scan over the edge table.
2. **ids** — one row per raw `cpty_id` with its dim attributes (name, type, FI, business / bank / placeholder flags).
3. **profile_{level}** — one row per unit: flow by direction, distinct PNC customers (all / TM / last month), active
   months, rails, number of raw ids, and the dominant member's attributes (by dollars) → `name_class`, `rail_group`, `role`.

`name_class`: from the map when it carries one (map v2: business / person / unknown / numeric_name / bank / placeholder /
no_name); otherwise the v1 dim flags with precedence placeholder → bank → no_name → business → person_other.

In [ ]:
# §2a  Window pairs (the expensive pass; reused if present)
def build_pairs():
    is_in = F.col("dst_kind") == "CPTY"                 # counterparty is the payee
    m_idx = (F.substring("month", 1, 4).cast("int") * 12 + F.substring("month", 6, 2).cast("int")) - F.lit(WIN_BASE)
    e = (spark.table(T_EDGE_M)
         .where(F.col("month").isin(WIN_MONTHS))
         .where((F.col("src_kind") == "CPTY") | (F.col("dst_kind") == "CPTY"))
         .select(F.when(is_in, F.col("dst_id")).otherwise(F.col("src_id")).alias("cpty_id"),
                 F.when(is_in, F.col("src_id")).otherwise(F.col("dst_id")).alias("cust"),
                 F.when(is_in, F.lit("in")).otherwise(F.lit("out")).alias("dir"),
                 m_idx.alias("m_idx"),
                 F.substring("category", 1, 1).cast("int").alias("rail_num"),   # 1-3 ACH, 4 wire, 5 RTP, 6 P2P, 7 check, 9 PCARD
                 F.coalesce(F.col("is_self"), F.lit(False)).alias("is_self"),
                 F.col("amount"), F.col("n_txn"),
                 (F.col("month") == LAST_MONTH).alias("is_lm")))
    pairs = e.groupBy("cpty_id", "cust", "dir").agg(
        F.sum("amount").alias("amt"),
        F.sum("n_txn").cast("long").alias("n_txn"),
        F.expr("bit_or(shiftleft(cast(1 as bigint), m_idx))").alias("mmask"),
        F.expr("bit_or(shiftleft(1, rail_num))").alias("rmask"),
        F.sum(F.when(F.col("is_self"), F.col("amount")).otherwise(F.lit(0))).alias("amt_self"),
        F.sum(F.when(F.col("is_lm"), F.col("amount")).otherwise(F.lit(0))).alias("amt_lm"),
        F.sum(F.when(F.col("is_lm"), F.col("n_txn")).otherwise(F.lit(0))).cast("long").alias("n_txn_lm"))
    emap = spark.table(T_MAP).select("cpty_id", "ent_l1", "ent_l2")
    cdim = spark.table(T_CUST_DIM).select(F.col("mdm_id").alias("cust"),
                                          F.col("is_tm").cast("boolean").alias("cust_is_tm"),
                                          F.lit(True).alias("cust_in_dim"))
    out = (pairs.join(emap, "cpty_id", "left")
                .join(cdim, "cust", "left")
                .withColumn("map_miss", F.col("ent_l2").isNull())
                .withColumn("ent_l1", F.coalesce(F.col("ent_l1"), F.col("cpty_id")))
                .withColumn("ent_l2", F.coalesce(F.col("ent_l2"), F.col("ent_l1")))
                .withColumn("cust_is_tm", F.coalesce(F.col("cust_is_tm"), F.lit(False)))
                .withColumn("cust_in_dim", F.coalesce(F.col("cust_in_dim"), F.lit(False))))
    out.write.mode("overwrite").parquet(P_PAIRS)

LOG = []
t0 = time.time()
rebuilt = FORCE or not path_done(P_PAIRS)
if rebuilt:
    build_pairs()
LOG.append({"step": "pairs", "action": "built" if rebuilt else "reused", "min": (time.time() - t0) / 60, "GB": path_gb(P_PAIRS)})

# Exact reconciliation against the source edges; a duplicate key in the map or customer dim fails here
src = (spark.table(T_EDGE_M).where(F.col("month").isin(WIN_MONTHS))
       .where((F.col("src_kind") == "CPTY") | (F.col("dst_kind") == "CPTY"))
       .agg(F.count("*").alias("edges"), F.sum("amount").alias("amt"), F.sum("n_txn").cast("long").alias("n_txn"),
            F.sum(((F.col("src_kind") == "CPTY") & (F.col("dst_kind") == "CPTY")).cast("int")).alias("cpty_cpty"))
       .first())
pairs = spark.read.parquet(P_PAIRS)
got = pairs.agg(F.count("*").alias("pairs"), F.sum("amt").alias("amt"), F.sum("n_txn").alias("n_txn"),
                F.sum(F.col("map_miss").cast("int")).alias("map_miss"),
                F.sum(F.when(F.col("map_miss"), F.col("amt")).otherwise(F.lit(0))).alias("map_miss_amt"),
                F.sum((~F.col("cust_in_dim")).cast("int")).alias("cust_miss"),
                F.sum(F.col("cust_is_tm").cast("int")).alias("tm_pairs"),
                F.sum(F.when(F.col("cust_is_tm"), F.col("amt")).otherwise(F.lit(0))).alias("tm_amt"),
                F.sum(F.when(F.col("dir") == "in", F.col("amt")).otherwise(F.lit(0))).alias("in_amt"),
                F.sum("amt_self").alias("self_amt"),
                F.sum("amt_lm").alias("lm_amt")).first()

assert src["cpty_cpty"] == 0, f"{src['cpty_cpty']:,} edges have a counterparty on both sides"
assert got["amt"] == src["amt"] and got["n_txn"] == src["n_txn"], (
    f"reconciliation failed: pairs ${got['amt']} / {got['n_txn']} txn vs edges ${src['amt']} / {src['n_txn']} txn")
miss_rate = got["map_miss"] / max(got["pairs"], 1)
assert miss_rate <= MAX_MAP_MISS, f"{miss_rate:.4%} of pairs have a cpty_id missing from {T_MAP}"

TOT_AMT = float(got["amt"])
show("§2a Window pairs — reconciled exactly with pkg_edge_monthly",
     ("Steps", pd.DataFrame(LOG)),
     ("Totals", pd.DataFrame([
         {"item": "counterparty edges in window", "value": f"{src['edges']:,}"},
         {"item": "pairs (cpty_id × customer × direction)", "value": f"{got['pairs']:,}"},
         {"item": "dollars", "value": f"${TOT_AMT / 1e9:,.2f}B"},
         {"item": "transactions", "value": f"{got['n_txn']:,}"},
         {"item": "% $ counterparty receives (in)", "value": f"{100 * float(got['in_amt']) / TOT_AMT:.2f}"},
         {"item": "% $ with TM customers", "value": f"{100 * float(got['tm_amt']) / TOT_AMT:.2f}"},
         {"item": "% $ self (own account elsewhere)", "value": f"{100 * float(got['self_amt']) / TOT_AMT:.2f}"},
         {"item": f"% $ in {LAST_MONTH}", "value": f"{100 * float(got['lm_amt']) / TOT_AMT:.2f}"},
         {"item": "pairs with cpty_id missing from map", "value": f"{got['map_miss']:,} (${float(got['map_miss_amt']):,.0f})"},
         {"item": "pairs with customer missing from pkg_cust_dim", "value": f"{got['cust_miss']:,}"}])))

In [ ]:
# §2b  Raw-id profile and unit profiles (reused if present)
def build_ids():
    p = spark.read.parquet(P_PAIRS)
    ids = p.groupBy("cpty_id", "ent_l1", "ent_l2").agg(F.sum("amt").alias("amt"), F.sum("n_txn").alias("n_txn"))
    d = spark.table(T_CPTY_DIM).select(
        "cpty_id",
        F.col(DIM["name"]).alias("name"),
        F.col(DIM["cpty_type"]).alias("cpty_type"),
        F.col(DIM["fi"]).alias("fi"),
        F.col(DIM["is_business"]).cast("boolean").alias("is_business"),
        F.col(DIM["name_is_fi"]).cast("boolean").alias("name_is_fi"),
        F.col(DIM["is_placeholder"]).cast("boolean").alias("is_placeholder"),
        (F.col(DIM["src"]) if DIM["src"] else F.lit(None).cast("string")).alias("src"),
        F.lit(True).alias("in_dim"))
    ids = ids.join(d, "cpty_id", "left").withColumn("in_dim", F.coalesce(F.col("in_dim"), F.lit(False)))
    if MAP_HAS_CLASS:      # map v2 carries typer v2 classes
        ids = ids.join(spark.table(T_MAP).select("cpty_id", F.col("name_class").alias("nc_map")), "cpty_id", "left")
    else:
        ids = ids.withColumn("nc_map", F.lit(None).cast("string"))
    ids.write.mode("overwrite").parquet(P_IDS)

RAIL_BITS = {"ACH": (1 << 1) | (1 << 2) | (1 << 3), "WIRE": 1 << 4, "RTP_PRT": 1 << 5,
             "RTP_P2P": 1 << 6, "CHECK": 1 << 7, "PCARD": 1 << 9}

def rail_group_col(c):
    expr = None
    for name, bits in RAIL_BITS.items():
        cond = (c.bitwiseAND(F.lit(~bits & 0x3FF)) == 0) & (c.bitwiseAND(F.lit(bits)) != 0)
        expr = F.when(cond, F.lit(f"{name} only")) if expr is None else expr.when(cond, F.lit(f"{name} only"))
    return expr.otherwise(F.lit("multi-rail"))

def build_profile(level):
    p = spark.read.parquet(P_PAIRS)
    in_, out_ = F.col("dir") == "in", F.col("dir") == "out"
    zero = F.lit(0)
    rel = p.groupBy(F.col(level).alias("unit_id"), "cust").agg(      # one row per (unit, PNC customer)
        F.sum("amt").alias("amt"),
        F.sum(F.when(in_, F.col("amt")).otherwise(zero)).alias("amt_in"),
        F.sum(F.when(out_, F.col("amt")).otherwise(zero)).alias("amt_out"),
        F.sum("n_txn").alias("n_txn"),
        F.sum(F.when(in_, F.col("n_txn")).otherwise(zero)).alias("n_txn_in"),
        F.sum(F.when(out_, F.col("n_txn")).otherwise(zero)).alias("n_txn_out"),
        F.sum("amt_self").alias("amt_self"),
        F.sum("amt_lm").alias("amt_lm"),
        F.sum("n_txn_lm").alias("n_txn_lm"),
        F.expr("bit_or(mmask)").alias("mmask"),
        F.expr("bit_or(rmask)").alias("rmask"),
        F.max(F.col("cust_is_tm").cast("int")).alias("tm"))
    sums = ["amt", "amt_in", "amt_out", "n_txn", "n_txn_in", "n_txn_out", "amt_self", "amt_lm", "n_txn_lm"]
    flow = rel.groupBy("unit_id").agg(
        F.count("*").alias("n_cust"),
        F.sum("tm").alias("n_cust_tm"),
        F.sum((F.col("n_txn_lm") > 0).cast("int")).alias("n_cust_lm"),
        F.sum((F.col("n_txn_in") > 0).cast("int")).alias("n_cust_in"),
        F.sum((F.col("n_txn_out") > 0).cast("int")).alias("n_cust_out"),
        *[F.sum(c).alias(c) for c in sums],
        F.sum(F.when(F.col("tm") == 1, F.col("amt")).otherwise(zero)).alias("amt_tm"),
        F.expr("bit_or(mmask)").alias("mmask"),
        F.expr("bit_or(rmask)").alias("rmask"))
    attr = spark.read.parquet(P_IDS).groupBy(F.col(level).alias("unit_id")).agg(
        F.count("*").alias("n_ids"),
        F.sum((~F.col("in_dim")).cast("int")).alias("n_ids_no_dim"),
        F.expr("max_by(struct(name, cpty_type, fi, src, is_business, name_is_fi, is_placeholder, nc_map), amt)").alias("dom"))
    nm = F.col("dom.name")
    prof = (flow.join(attr, "unit_id", "left")
            .withColumn("n_months", F.expr("bit_count(mmask)"))
            .withColumn("name", nm)
            .withColumn("cpty_type", F.coalesce(F.col("dom.cpty_type"), F.lit("(null)")))
            .withColumn("fi", F.col("dom.fi"))
            .withColumn("src", F.coalesce(F.col("dom.src"), F.lit("(null)")))
            .withColumn("name_class",
                        F.when(F.col("dom.nc_map").isNotNull(), F.col("dom.nc_map"))
                         .when(F.col("dom.is_placeholder"), "placeholder")
                         .when(F.col("dom.name_is_fi"), "bank")
                         .when(nm.isNull() | (F.trim(nm) == ""), "no_name")
                         .when(F.col("dom.is_business"), "business")
                         .otherwise("person_other"))
            .withColumn("rail_group", rail_group_col(F.col("rmask")))
            .withColumn("role", F.when((F.col("n_txn_in") > 0) & (F.col("n_txn_out") > 0), "both")
                                 .when(F.col("n_txn_in") > 0, "payee").otherwise("payer"))
            .withColumn("tm_touch", F.col("n_cust_tm") > 0)
            .drop("dom"))
    prof.write.mode("overwrite").parquet(P_PROF(level))

t0 = time.time()
rebuilt = rebuilt or not path_done(P_IDS)
if rebuilt:
    build_ids()
LOG.append({"step": "ids", "action": "built" if rebuilt else "reused", "min": (time.time() - t0) / 60, "GB": path_gb(P_IDS)})

ids = spark.read.parquet(P_IDS)
ic = ids.agg(F.count("*").alias("rows"), F.countDistinct("cpty_id").alias("ids"), F.sum("amt").alias("amt"),
             F.sum((~F.col("in_dim")).cast("int")).alias("no_dim"),
             F.countDistinct("ent_l1").alias("ent_l1"), F.countDistinct("ent_l2").alias("ent_l2")).first()
assert ic["rows"] == ic["ids"], f"pkg_cpty_dim has duplicate cpty_id: {ic['rows']:,} rows for {ic['ids']:,} ids"
assert ic["amt"] == got["amt"], "id profile does not reconcile with pairs"
assert ic["no_dim"] / ic["ids"] <= MAX_DIM_MISS, f"{ic['no_dim']:,} window ids missing from {T_CPTY_DIM}"

PROF_CHECK = []
for level in LEVELS:
    t0 = time.time()
    built = rebuilt or not path_done(P_PROF(level))
    if built:
        build_profile(level)
    LOG.append({"step": f"profile_{level}", "action": "built" if built else "reused",
                "min": (time.time() - t0) / 60, "GB": path_gb(P_PROF(level))})
    pc = spark.read.parquet(P_PROF(level)).agg(
        F.count("*").alias("units"), F.sum("amt").alias("amt"), F.sum("n_txn").alias("n_txn"),
        F.sum("n_ids").alias("ids"), F.sum(F.col("n_ids").isNull().cast("int")).alias("no_attr")).first()
    assert pc["amt"] == got["amt"] and pc["n_txn"] == got["n_txn"], f"profile_{level} does not reconcile"
    assert pc["ids"] == ic["ids"] and pc["no_attr"] == 0, f"profile_{level}: id attribution broken"
    PROF_CHECK.append({"level": LEVEL_NAME.get(level, level), "units": pc["units"], "raw ids": pc["ids"],
                       "ids per unit": pc["ids"] / pc["units"]})

show("§2b Profiles — reconciled with the pair table",
     ("Steps", pd.DataFrame(LOG)),
     ("Units", pd.DataFrame(PROF_CHECK)),
     note(f"Window ids: {ic['ids']:,} raw → {ic['ent_l1']:,} ent_l1 → {ic['ent_l2']:,} ent_l2; "
          f"{ic['no_dim']:,} ids missing from pkg_cpty_dim."))

## §3 Headline numbers

One pass per unit level builds the **cube**: unit counts and sums (dollars, transactions, relationships, TM dollars,
TM relationships) by grid position on transactions × amount × PNC customers × active months, plus `name_class` and
whether the unit touches a TM customer. Every rule on the grids is evaluated exactly from it in pandas. Candidate rules
are also evaluated directly in Spark, so off-grid thresholds work too.

A *relationship* is one (unit, PNC customer) pair; TM relationships are those with a TM customer.

In [ ]:
# §3  Cube, quantiles and candidate rules per level
METRICS = ["ents", "ids", "amt", "n_txn", "rels", "amt_tm", "rels_tm"]

def cube_of(prof):
    return (prof.groupBy(idx_col("n_txn", K_GRID).alias("i_n"), idx_col("amt", A_GRID).alias("i_a"),
                         idx_col("n_cust", D_GRID).alias("i_d"), idx_col("n_months", M_GRID).alias("i_m"),
                         "name_class", "tm_touch")
                .agg(F.count("*").alias("ents"), F.sum("n_ids").alias("ids"),
                     F.sum(F.col("amt").cast("double")).alias("amt"), F.sum("n_txn").alias("n_txn"),
                     F.sum("n_cust").alias("rels"), F.sum(F.col("amt_tm").cast("double")).alias("amt_tm"),
                     F.sum("n_cust_tm").alias("rels_tm"), F.sum(F.col("amt_self").cast("double")).alias("amt_self"))
                .toPandas())

QS = [0.5, 0.75, 0.9, 0.99]
def quantiles_of(prof):
    r = prof.agg(*[F.expr(f"percentile_approx(cast({c} as double), array({','.join(map(str, QS))}))").alias(c)
                   for c in ["n_txn", "amt", "n_cust", "n_months"]]).first()
    return {c: list(r[c]) for c in ["n_txn", "amt", "n_cust", "n_months"]}

def metric_cols(sfx=""):
    if sfx:      # single-month weights
        return {"ents": F.lit(1).cast("long"), "amt": F.col("amt" + sfx).cast("double"),
                "n_txn": F.col("n_txn" + sfx).cast("long"), "rels": F.col("n_cust" + sfx).cast("long")}
    return {"ents": F.lit(1).cast("long"), "ids": F.col("n_ids").cast("long"), "amt": F.col("amt").cast("double"),
            "n_txn": F.col("n_txn").cast("long"), "rels": F.col("n_cust").cast("long"),
            "amt_tm": F.col("amt_tm").cast("double"), "rels_tm": F.col("n_cust_tm").cast("long")}

def eval_rules(df, rules, metrics, rule_sfx="", by="name_class"):
    # One pass: conditional sums of each metric for every rule, grouped by `by`. Returns a long pandas frame.
    aggs = [F.sum(c).alias(f"tot__{m}") for m, c in metrics.items()]
    for i, r in enumerate(rules):
        rc = rule_col(r, rule_sfx)
        for m, c in metrics.items():
            aggs.append((F.sum(F.when(rc, c)) if rc is not None else F.lit(None).cast("double")).alias(f"r{i}__{m}"))
    pdf = (df.groupBy(by) if by else df).agg(*aggs).toPandas()
    rows = []
    for _, row in pdf.iterrows():
        for i, r in enumerate(rules):
            defined = rule_col(r, rule_sfx) is not None
            rec = {"rule_i": i, "rule": cand_name(i) if rules is CANDIDATE_RULES else rule_label(r), "defined": defined}
            if by:
                rec[by] = row[by]
            for m in metrics:
                k = row[f"r{i}__{m}"]
                rec[f"kept_{m}"] = (0.0 if pd.isna(k) else float(k)) if defined else np.nan
                rec[f"tot_{m}"] = float(row[f"tot__{m}"])
            rows.append(rec)
    return pd.DataFrame(rows)

RES = {}
for level in LEVELS:
    t0 = time.time()
    prof = spark.read.parquet(P_PROF(level))
    RES[level] = {"cube": cube_of(prof), "q": quantiles_of(prof),
                  "cand": eval_rules(prof, CANDIDATE_RULES, metric_cols())}
    RES[level]["min"] = (time.time() - t0) / 60

PRIMARY = LEVELS[0]
CUBE = RES[PRIMARY]["cube"]

def headline(level):
    cube, q = RES[level]["cube"], RES[level]["q"]
    tot = cube[METRICS].sum()
    pct = lambda m, mask: 100 * cube.loc[mask, m].sum() / tot[m]
    n1 = cube.i_n == 1
    small = cube.i_a <= A_GRID.index(100)          # amount < $100
    return {
        "units": f"{tot.ents:,.0f}",
        "raw ids": f"{tot.ids:,.0f}",
        "dollars": f"${tot.amt / 1e9:,.2f}B",
        "transactions": f"{tot.n_txn:,.0f}",
        "relationships (unit × PNC customer)": f"{tot.rels:,.0f}",
        "% units with 1 transaction": f"{pct('ents', n1):.2f}",
        "% units with 1 transaction and < $100": f"{pct('ents', n1 & small):.2f}",
        "% $ held by 1-transaction units": f"{pct('amt', n1):.3f}",
        "% units with 1 PNC customer": f"{pct('ents', cube.i_d == 1):.2f}",
        "% units active in 1 month": f"{pct('ents', cube.i_m == 1):.2f}",
        "% units touching a TM customer": f"{pct('ents', cube.tm_touch):.2f}",
        "transactions p50 / p75 / p90 / p99": " / ".join(f"{v:,.0f}" for v in q["n_txn"]),
        "dollars p50 / p75 / p90 / p99": " / ".join(usd(round(v)) for v in q["amt"]),
        "PNC customers p50 / p90 / p99": " / ".join(f"{v:,.0f}" for v in (q["n_cust"][0], q["n_cust"][2], q["n_cust"][3])),
        "build time (min)": f"{RES[level]['min']:.1f}",
    }

hl = pd.DataFrame({LEVEL_NAME.get(l, l): headline(l) for l in LEVELS})
hl.insert(0, "measure", hl.index)
show(f"§3 Headline — window {WIN_START} → {LAST_MONTH}", ("Per unit level", hl.reset_index(drop=True)),
     note(f"Cube for {LEVEL_NAME.get(PRIMARY, PRIMARY)}: {len(CUBE):,} cells. "
          "Quantiles are approximate (percentile_approx); every share is exact."))

## §4 Distributions

Bucket tables for the primary unit. `cum` columns are cumulative from the smallest bucket: the share of units, or of
dollars, **at or below** that bucket. Reading across the first rows answers the core question directly — e.g. how
many units sit at 1–2 transactions and how little money they hold.

Charts (saved to `OUT_DIR`): `ccdf_all.png` — share of units with a value ≥ x (solid) and share of dollars held by
those units (dashed); `ccdf_by_class.png` — the unit share by `name_class`.

In [ ]:
# §4  Bucket tables
def bucket_table(cube, icol, grid, money=False):
    labs = grid_labels(grid, money)
    g = cube.groupby(icol)[METRICS].sum().sort_index()
    biz = cube[cube.name_class == "business"].groupby(icol)["ents"].sum().reindex(g.index, fill_value=0)
    tm = cube[cube.tm_touch].groupby(icol)["ents"].sum().reindex(g.index, fill_value=0)
    tot = g.sum()
    return pd.DataFrame({
        "bucket": [labs[i] for i in g.index],
        "units": g["ents"].values,
        "% units": 100 * g["ents"].values / tot["ents"],
        "cum % units": 100 * g["ents"].cumsum().values / tot["ents"],
        "% $": 100 * g["amt"].values / tot["amt"],
        "cum % $": 100 * g["amt"].cumsum().values / tot["amt"],
        "% txn": 100 * g["n_txn"].values / tot["n_txn"],
        "% relationships": 100 * g["rels"].values / tot["rels"],
        "% TM relationships": 100 * g["rels_tm"].values / tot["rels_tm"],
        "% of bucket business": 100 * biz.values / g["ents"].values,
        "% of bucket touching TM": 100 * tm.values / g["ents"].values,
    })

show(f"§4 Distributions — {LEVEL_NAME.get(PRIMARY, PRIMARY)}, window {WIN_START} → {LAST_MONTH}",
     ("Transactions in window", bucket_table(CUBE, "i_n", K_GRID)),
     ("Dollars in window", bucket_table(CUBE, "i_a", A_GRID, money=True)),
     ("Distinct PNC customers", bucket_table(CUBE, "i_d", D_GRID)),
     ("Active months", bucket_table(CUBE, "i_m", M_GRID)))

In [ ]:
# §4  CCDF charts (fine log bins; saved to disk)
def fine_bins(prof, col, money):
    x = F.col(col).cast("double")
    lg = F.pow(F.lit(10.0), F.floor(F.log10(x) * 20) / 20)          # 20 bins per decade
    b = F.when(x < 1, F.lit(0.0)).otherwise(lg) if money else F.when(x <= 100, x).otherwise(lg)
    return (prof.groupBy(b.alias("bin"), "name_class")
                .agg(F.count("*").alias("ents"), F.sum(F.col("amt").cast("double")).alias("amt"))
                .toPandas())

prof = spark.read.parquet(P_PROF(PRIMARY))
VARS = [("n_txn", "transactions in window", False), ("amt", "dollars in window", True),
        ("n_cust", "distinct PNC customers", False), ("n_months", "active months", False)]
FINE = {c: fine_bins(prof, c, money) for c, _, money in VARS}

def ccdf(df):
    g = df.groupby("bin")[["ents", "amt"]].sum().sort_index()
    rev = g[::-1].cumsum()[::-1]
    return g.index.to_numpy(), (rev["ents"] / g["ents"].sum()).to_numpy(), (rev["amt"] / g["amt"].sum()).to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
for ax, (c, label, money) in zip(axes.ravel(), VARS):
    x, pe, pa = ccdf(FINE[c])
    keep = x > 0
    ax.plot(x[keep], pe[keep], color=C_BLUE, label="share of units with value ≥ x")
    ax.plot(x[keep], pa[keep], color=C_ORANGE, linestyle="--", label="share of dollars in those units")
    if c != "n_months":
        ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(label)
axes[0, 0].legend(loc="lower left")
fig.suptitle(f"Counterparty {LEVEL_NAME.get(PRIMARY, PRIMARY)} — CCDF, {WIN_START} → {LAST_MONTH}")
p1 = save_fig(fig, "ccdf_all.png")

_cls_order = FINE["n_txn"].groupby("name_class")["ents"].sum().sort_values(ascending=False).index.tolist()[:5]
CLASS_COLORS = list(zip(_cls_order, [C_BLUE, C_ORANGE, C_AQUA, C_YELLOW, C_GREY]))
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
for ax, (c, label, money) in zip(axes.ravel(), VARS):
    for cls, col in CLASS_COLORS:
        d = FINE[c][FINE[c].name_class == cls]
        if d.empty:
            continue
        x, pe, _ = ccdf(d)
        keep = x > 0
        ax.plot(x[keep], pe[keep], color=col, label=f"{cls} ({d.ents.sum():,.0f})")
    if c != "n_months":
        ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(label)
    ax.set_ylabel("share of units with value ≥ x")
axes[0, 0].legend(loc="lower left")
fig.suptitle(f"CCDF by name_class — {LEVEL_NAME.get(PRIMARY, PRIMARY)}")
p2 = save_fig(fig, "ccdf_by_class.png")

show("§4 Charts saved", note(p1), note(p2))

## §5 Joint distribution and the single-transaction tail

The joint table shows where units and dollars sit on transactions × amount together — the region a combined rule
would cut. The single-transaction tables say *what* the one-off units are: which rails, roles and counterparty types
produce them.

In [ ]:
# §5  Joint distribution (coarse) and single-transaction tail
N_COARSE = [1, 2, 3, 5, 10, 25, 100]
A_COARSE = [0, 100, 1_000, 10_000, 100_000, 1_000_000]
assert set(N_COARSE) <= set(K_GRID) and set(A_COARSE) <= set(A_GRID)

def to_coarse(fine_idx, grid, coarse):
    lo = np.array([grid[i - 1] if i >= 1 else -np.inf for i in fine_idx], dtype=float)
    return np.searchsorted(np.array(coarse, dtype=float), lo, side="right")

def joint(cube, metric):
    c = cube.assign(cn=to_coarse(cube.i_n, K_GRID, N_COARSE), ca=to_coarse(cube.i_a, A_GRID, A_COARSE))
    pv = c.pivot_table(index="cn", columns="ca", values=metric, aggfunc="sum", fill_value=0)
    pv = 100 * pv / pv.values.sum()
    nl, al = grid_labels(N_COARSE), grid_labels(A_COARSE, money=True)
    pv.index = [f"txn {nl[i]}" for i in pv.index]
    pv.columns = [al[j] for j in pv.columns]
    pv["row total"] = pv.sum(axis=1)
    return pv.reset_index().rename(columns={"index": "transactions"})

prof = spark.read.parquet(P_PROF(PRIMARY))
tail = (prof.groupBy((F.col("n_txn") == 1).alias("one_txn"), idx_col("amt", A_COARSE).alias("ca"),
                     "name_class", "rail_group", "role", "cpty_type", "src")
            .agg(F.count("*").alias("ents"), F.sum(F.col("amt").cast("double")).alias("amt"))
            .toPandas())

def tail_by(dim):
    g = tail.groupby([dim, "one_txn"])[["ents", "amt"]].sum().unstack("one_txn", fill_value=0)
    get = lambda m, k: g[(m, k)].to_numpy(float) if (m, k) in g.columns else np.zeros(len(g))
    e1, e0, a1 = get("ents", True), get("ents", False), get("amt", True)
    t = pd.DataFrame({dim: g.index.astype(str), "units": e1 + e0, "1-txn units": e1,
                      "% of segment 1-txn": 100 * e1 / (e1 + e0),
                      "% of all 1-txn units": 100 * e1 / max(e1.sum(), 1),
                      "$M in 1-txn units": a1 / 1e6})
    return t.sort_values("1-txn units", ascending=False)

one = tail[tail.one_txn]
al = grid_labels(A_COARSE, money=True)
amt_cls = one.pivot_table(index="ca", columns="name_class", values="ents", aggfunc="sum", fill_value=0)
amt_cls = 100 * amt_cls / amt_cls.values.sum()
amt_cls.index = [al[i] for i in amt_cls.index]
amt_cls["all"] = amt_cls.sum(axis=1)

show(f"§5 Joint distribution and single-transaction tail — {LEVEL_NAME.get(PRIMARY, PRIMARY)}",
     ("% of units by transactions × dollars", joint(CUBE, "ents"), "pct"),
     ("% of dollars by transactions × dollars", joint(CUBE, "amt"), "pct"),
     ("% of TM relationships by transactions × dollars", joint(CUBE, "rels_tm"), "pct"),
     ("Single-transaction units: % by amount × name_class", amt_cls.reset_index().rename(columns={"index": "amount"}), "pct"),
     ("Single-transaction units by name_class", tail_by("name_class")),
     ("… by rail group", tail_by("rail_group")),
     ("… by role (payee = receives from PNC customers)", tail_by("role")),
     ("… by cpty_type", tail_by("cpty_type")),
     ("… by id source (A = account, B = name hash)", tail_by("src")))

## §6 Threshold trade-off

- **Candidate rules** (exact, any thresholds): share of units kept against the share of value dropped. The value
  columns are the cost: dollars, transactions, relationships, and the TM-customer subset.
- **k × $ grids** from the cube: AND rules (row `txn≥1` is a dollar-only rule, column `amt≥$0` a transaction-only
  rule) and OR rules.
- **Rule search**: every grid rule in six families, filtered by `KEEP_FLOORS`, ranked by fewest units kept.

Charts: `grid_and.png`, `grid_or.png` (heatmaps), `frontier.png` (units kept vs value dropped for every grid rule).

In [ ]:
# §6a  Candidate rules
def cand_table(ev):
    ms = ["ents", "amt", "n_txn", "rels", "amt_tm", "rels_tm"]
    g = ev.groupby(["rule_i", "rule"])[[f"kept_{m}" for m in ms] + [f"tot_{m}" for m in ms]].sum(min_count=1).reset_index()
    out = pd.DataFrame({"rule": g["rule"], "units kept": g["kept_ents"],
                        "% units kept": 100 * g["kept_ents"] / g["tot_ents"]})
    for m, lab in [("amt", "$"), ("n_txn", "txn"), ("rels", "relationships"), ("amt_tm", "TM $"), ("rels_tm", "TM relationships")]:
        out[f"% {lab} dropped"] = 100 * (1 - g[f"kept_{m}"] / g[f"tot_{m}"])
    return out

def cand_by_class(ev, metric, kept=True):
    g = ev.groupby(["rule_i", "rule", "name_class"])[[f"kept_{metric}", f"tot_{metric}"]].sum().reset_index()
    v = g[f"kept_{metric}"] / g[f"tot_{metric}"]
    g["v"] = 100 * (v if kept else 1 - v)
    pv = g.pivot_table(index=["rule_i", "rule"], columns="name_class", values="v").reset_index().drop(columns="rule_i")
    return pv

ev = RES[PRIMARY]["cand"]
sizes = ev[ev.rule_i == 0].set_index("name_class")[["tot_ents", "tot_amt"]]
sizes = pd.DataFrame({"name_class": sizes.index, "units": sizes.tot_ents.values,
                      "% units": 100 * sizes.tot_ents.values / sizes.tot_ents.sum(),
                      "% $": 100 * sizes.tot_amt.values / sizes.tot_amt.sum()}).sort_values("units", ascending=False)
show(f"§6a Candidate rules — {LEVEL_NAME.get(PRIMARY, PRIMARY)}, window {WIN_START} → {LAST_MONTH}",
     ("Units kept vs value dropped", cand_table(ev)),
     ("Size of each name_class", sizes),
     ("% of units kept, by name_class", cand_by_class(ev, "ents"), "pct"),
     ("% of $ dropped, by name_class", cand_by_class(ev, "amt", kept=False), "pct"))

In [ ]:
# §6b  k × $ grids from the cube (inline subset; full grids in the heatmaps)
_vals = CUBE[METRICS].to_numpy(float)
_tot = _vals.sum(axis=0)
MI = {m: i for i, m in enumerate(METRICS)}

def shares(r):
    return _vals[cube_mask(CUBE, r)].sum(axis=0) / _tot

def grid_pivot(ks, amts, logic, metric, kept):
    rows = []
    for k in ks:
        rec = {"rule": f"txn≥{k}"}
        for a in amts:
            s = shares({"n": k, "amt": a, "logic": logic})[MI[metric]]
            rec[f"{'AND' if logic == 'and' else 'OR'} amt≥{usd(a)}"] = 100 * (s if kept else 1 - s)
        rows.append(rec)
    return pd.DataFrame(rows)

KS_AND, AS_AND = [1, 2, 3, 5, 10, 20], [0, 100, 500, 1_000, 5_000, 10_000, 50_000]
KS_OR,  AS_OR  = [2, 3, 5, 10, 20, 50], [1_000, 2_500, 5_000, 10_000, 25_000, 50_000, 100_000]
show(f"§6b k × $ grids — {LEVEL_NAME.get(PRIMARY, PRIMARY)}",
     ("AND — % of units kept", grid_pivot(KS_AND, AS_AND, "and", "ents", True), "pct"),
     ("AND — % of $ dropped", grid_pivot(KS_AND, AS_AND, "and", "amt", False), "pct"),
     ("AND — % of TM relationships dropped", grid_pivot(KS_AND, AS_AND, "and", "rels_tm", False), "pct"),
     ("OR — % of units kept", grid_pivot(KS_OR, AS_OR, "or", "ents", True), "pct"),
     ("OR — % of $ dropped", grid_pivot(KS_OR, AS_OR, "or", "amt", False), "pct"),
     ("OR — % of TM relationships dropped", grid_pivot(KS_OR, AS_OR, "or", "rels_tm", False), "pct"))

In [ ]:
# §6c  Rule search under KEEP_FLOORS, and charts
# Simplest families first: when two rules keep exactly the same units, the first one listed is reported
FAMILIES = (
    [("txn only", {"n": k}) for k in K_GRID[1:]]
    + [("amt only", {"amt": a}) for a in A_GRID[1:]]
    + [("months only", {"months": m}) for m in M_GRID[1:]]
    + [("txn AND amt", {"n": k, "amt": a, "logic": "and"}) for k in K_GRID[1:] for a in A_GRID[1:]]
    + [("txn OR amt", {"n": k, "amt": a}) for k in K_GRID[1:] for a in A_GRID[1:]]
    + [("cust OR amt", {"cust": d, "amt": a}) for d in D_GRID[1:] for a in A_GRID[1:]]
    + [("months OR amt", {"months": m, "amt": a}) for m in M_GRID[1:] for a in A_GRID[1:]]
    + [("txn OR cust", {"n": k, "cust": d}) for k in K_GRID[1:] for d in D_GRID[1:]])

SEARCH = pd.DataFrame([{"family": fam, "rule": rule_label(r), "rule_obj": json.dumps(r),
                        **{f"keep_{m}": v for m, v in zip(METRICS, shares(r))}} for fam, r in FAMILIES])
_sig = SEARCH[[f"keep_{m}" for m in METRICS]].round(12).apply(tuple, axis=1)
UNIQ = SEARCH[~_sig.duplicated(keep="first")]                     # drop rules equivalent to a simpler one
ok = np.ones(len(UNIQ), bool)
for m, floor in KEEP_FLOORS.items():
    ok &= UNIQ[f"keep_{m}"].to_numpy() >= floor
PASS = UNIQ[ok].sort_values(["keep_ents", "keep_amt"], ascending=[True, False])

def search_view(df):
    out = pd.DataFrame({"family": df.family, "rule": df.rule, "% units kept": 100 * df.keep_ents})
    for m, lab in [("amt", "$"), ("n_txn", "txn"), ("rels", "relationships"), ("amt_tm", "TM $"), ("rels_tm", "TM relationships")]:
        out[f"% {lab} dropped"] = 100 * (1 - df[f"keep_{m}"])
    return out

best_per_family = PASS.groupby("family", sort=False).head(1).sort_values("keep_ents")

# ---- heatmaps ----
def heat(logic, fname):
    ks = K_GRID if logic == "and" else K_GRID[1:]
    amts = A_GRID if logic == "and" else A_GRID[1:]
    panels = [("ents", True, "% of units kept", "Blues"), ("amt", False, "% of $ dropped", "Oranges"),
              ("rels_tm", False, "% of TM relationships dropped", "Oranges")]
    fig, axes = plt.subplots(1, 3, figsize=(21, 6.5), constrained_layout=True)
    for ax, (m, kept, title, cmap) in zip(axes, panels):
        z = np.array([[100 * (shares({"n": k, "amt": a, "logic": logic})[MI[m]] if kept
                              else 1 - shares({"n": k, "amt": a, "logic": logic})[MI[m]])
                       for a in amts] for k in ks])
        ax.imshow(z, cmap=cmap, aspect="auto")
        ax.grid(False)
        ax.set_xticks(range(len(amts)))
        ax.set_xticklabels([usd(a) for a in amts], rotation=60)
        ax.set_yticks(range(len(ks)))
        ax.set_yticklabels([f"{k:,}" for k in ks])
        ax.set_xlabel("amount threshold")
        ax.set_ylabel("transaction threshold")
        ax.set_title(title)
        hi = np.nanmax(z) if np.isfinite(z).any() else 1
        for i in range(len(ks)):
            for j in range(len(amts)):
                v = z[i, j]
                ax.text(j, i, f"{v:.1f}" if v >= 1 else f"{v:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if v > 0.6 * hi else "#0b0b0b")
    fig.suptitle(f"txn {'AND' if logic == 'and' else 'OR'} amount — {LEVEL_NAME.get(PRIMARY, PRIMARY)}, {WIN_START} → {LAST_MONTH}")
    return save_fig(fig, fname)

p_and, p_or = heat("and", "grid_and.png"), heat("or", "grid_or.png")

# ---- frontier ----
FAM_STYLE = {"txn OR amt": (C_BLUE, "o"), "txn AND amt": (C_ORANGE, "s"), "cust OR amt": (C_AQUA, "^"),
             "months OR amt": (C_YELLOW, "D"), "txn OR cust": (C_GREY, "v"),
             "txn only": ("#0b0b0b", "x"), "amt only": ("#0b0b0b", "+"), "months only": ("#0b0b0b", "1")}
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), constrained_layout=True)
for ax, (m, lab, logy) in zip(axes, [("amt", "% of $ dropped", True), ("rels_tm", "% of TM relationships dropped", False)]):
    for fam, (col, mk) in FAM_STYLE.items():
        d = UNIQ[UNIQ.family == fam]
        y = 100 * (1 - d[f"keep_{m}"])
        ax.scatter(100 * d.keep_ents, np.maximum(y, 1e-4) if logy else y, s=14, color=col, marker=mk,
                   alpha=0.8, label=fam)
    for i, r in enumerate(CANDIDATE_RULES):
        if r and on_grid(r):
            s = shares(r)
            x, y = 100 * s[MI["ents"]], 100 * (1 - s[MI[m]])
            y = max(y, 1e-4) if logy else y
            ax.scatter([x], [y], s=70, facecolors="none", edgecolors="#0b0b0b", linewidths=1.3)
            ax.annotate(f"R{i}", (x, y), fontsize=8, fontweight="bold" if i == CHOSEN_I else "normal",
                        xytext=(5, 4), textcoords="offset points")
    ax.set_xscale("log")
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("% of units kept")
    ax.set_ylabel(lab + (" (floored at 0.0001)" if logy else ""))
    ax.set_title(f"units kept vs {lab}")
axes[0].legend(loc="lower left", fontsize=8)
fig.suptitle(f"Every grid rule — {LEVEL_NAME.get(PRIMARY, PRIMARY)}; circled = candidate rules (ids as in §6a)")
p_front = save_fig(fig, "frontier.png")

show(f"§6c Rule search — floors {', '.join(f'{k} ≥ {v:.1%}' for k, v in KEEP_FLOORS.items())}",
     note(f"{len(SEARCH):,} grid rules searched ({len(UNIQ):,} distinct); {ok.sum():,} meet the floors."),
     ("Best rule per family (fewest units kept)", search_view(best_per_family)),
     ("Top 15 overall", search_view(PASS.head(15))),
     note(f"Charts: {p_and} | {p_or} | {p_front}"))

## §7 Chosen rule — what gets dropped

Composition of kept and dropped units for `CHOSEN_RULE`, then examples to eyeball: the largest dropped units, the
largest single-transaction units (kept only by the dollar leg if the rule has one), a random sample of dropped
businesses, and kept units sitting exactly at the transaction threshold.

In [ ]:
# §7a  Composition
prof = spark.read.parquet(P_PROF(PRIMARY))
kept_c = rule_col(CHOSEN_RULE)
comp = (prof.groupBy(kept_c.alias("kept"), "name_class", "rail_group", "role", "cpty_type", "src", "tm_touch")
            .agg(F.count("*").alias("ents"), F.sum(F.col("amt").cast("double")).alias("amt"),
                 F.sum("n_cust_tm").alias("rels_tm"))
            .toPandas())

def comp_table(dim):
    g = comp.groupby([dim, "kept"])[["ents", "amt", "rels_tm"]].sum().unstack("kept", fill_value=0)
    get = lambda m, k: g[(m, k)].to_numpy(float) if (m, k) in g.columns else np.zeros(len(g))
    ek, ed, ad, at = get("ents", True), get("ents", False), get("amt", False), get("amt", True) + get("amt", False)
    rd, rt = get("rels_tm", False), get("rels_tm", True) + get("rels_tm", False)
    out = pd.DataFrame({dim: g.index.astype(str), "units": ek + ed, "kept": ek,
                        "% of segment kept": 100 * ek / (ek + ed),
                        "% of all dropped": 100 * ed / ed.sum() if ed.sum() else 0.0,
                        "$M dropped": ad / 1e6,
                        "% of segment $ dropped": 100 * np.divide(ad, at, out=np.zeros_like(ad), where=at > 0),
                        "% of segment TM rel. dropped": 100 * np.divide(rd, rt, out=np.zeros_like(rd), where=rt > 0)})
    return out.sort_values("units", ascending=False)

tot_k, tot = comp.loc[comp.kept, "ents"].sum(), comp.ents.sum()
show(f"§7a Chosen rule {rule_label(CHOSEN_RULE)} — {LEVEL_NAME.get(PRIMARY, PRIMARY)}",
     note(f"Keeps {tot_k:,.0f} of {tot:,.0f} units ({100 * tot_k / tot:.2f}%), "
          f"{100 * comp.loc[comp.kept, 'amt'].sum() / comp.amt.sum():.3f}% of dollars."),
     ("By name_class", comp_table("name_class")),
     ("By rail group", comp_table("rail_group")),
     ("By role", comp_table("role")),
     ("By cpty_type", comp_table("cpty_type")),
     ("By id source", comp_table("src")),
     ("By TM contact", comp_table("tm_touch")))

In [ ]:
# §7b  Examples
EX_COLS = ["name", "name_class", "cpty_type", "fi", "n_ids", "n_txn", "amt", "n_cust", "n_cust_tm", "n_months",
           "rail_group", "role"]

def ex(df, n=20):
    pdf = df.select(*EX_COLS).limit(n).toPandas()
    pdf["amt"] = pdf["amt"].astype(float)
    return pdf.rename(columns={"amt": "$ amount", "n_ids": "ids", "n_txn": "txn", "n_cust": "PNC cust",
                               "n_cust_tm": "TM cust", "n_months": "months"})

dropped = prof.where(~kept_c)
n_drop_biz = dropped.where(F.col("name_class") == "business").count()
frac = min(1.0, 60.0 / max(n_drop_biz, 1))
k_at = CHOSEN_RULE.get("n")
blocks = [f"§7b Examples — {rule_label(CHOSEN_RULE)}",
          ("Largest dropped units by dollars", ex(dropped.orderBy(F.desc("amt")))),
          ("Largest single-transaction units", ex(prof.where(F.col("n_txn") == 1).orderBy(F.desc("amt")))),
          ("Random dropped businesses", ex(dropped.where(F.col("name_class") == "business")
                                             .where(F.rand(7) < frac)))]
if k_at:
    # OR rule with a dollar leg: units kept only because they reached k transactions
    or_amt = "amt" in CHOSEN_RULE and CHOSEN_RULE.get("logic", "or") == "or"
    at_k = prof.where((F.col("n_txn") == k_at) & kept_c)
    if or_amt:
        at_k = at_k.where(F.col("amt") < CHOSEN_RULE["amt"])
    n_at = at_k.count()
    blocks.append((f"Random kept units at exactly txn = {k_at}" + (f" and below {usd(CHOSEN_RULE['amt'])}" if or_amt else "")
                   + f" ({n_at:,} such units)",
                   ex(at_k.where(F.rand(11) < min(1.0, 60.0 / max(n_at, 1))))))
show(*blocks)

## §8 Window vs single month

Counterparty metrics will be monthly, so the question is which units in a given month get the full set. Population:
units active in `LAST_MONTH`; weights are that month's dollars, transactions and relationships. Two ways to decide
eligibility:

- **window rule** — the rule applied to the trailing-window totals (what this notebook proposes);
- **month rule** — the same rule applied to that month alone (months-based rules have no single-month meaning).

If the month rule drops far more units *and* far more of the month's dollars, a trailing window is the right basis.

In [ ]:
# §8  Window rule vs month rule on last-month activity
prof = spark.read.parquet(P_PROF(PRIMARY)).where(F.col("n_txn_lm") > 0)
ev_w = eval_rules(prof, CANDIDATE_RULES, metric_cols("_lm"), rule_sfx="", by=None)
ev_m = eval_rules(prof, CANDIDATE_RULES, metric_cols("_lm"), rule_sfx="_lm", by=None)

rows = []
for (_, w), (_, m) in zip(ev_w.iterrows(), ev_m.iterrows()):
    rec = {"rule": w["rule"]}
    for lab, e in [("window", w), ("month", m)]:
        rec[f"% units kept ({lab})"] = 100 * e["kept_ents"] / e["tot_ents"]
    for lab, e in [("window", w), ("month", m)]:
        rec[f"% month $ dropped ({lab})"] = 100 * (1 - e["kept_amt"] / e["tot_amt"])
    for lab, e in [("window", w), ("month", m)]:
        rec[f"% month relationships dropped ({lab})"] = 100 * (1 - e["kept_rels"] / e["tot_rels"])
    rows.append(rec)

lm_cube = (prof.groupBy(idx_col("n_txn_lm", K_GRID).alias("i_n"))
               .agg(F.count("*").alias("ents"), F.sum(F.col("amt_lm").cast("double")).alias("amt"))
               .toPandas().sort_values("i_n"))
labs = grid_labels(K_GRID)
lm_tab = pd.DataFrame({"transactions in month": [labs[i] for i in lm_cube.i_n], "units": lm_cube.ents,
                       "% units": 100 * lm_cube.ents / lm_cube.ents.sum(),
                       "cum % units": 100 * lm_cube.ents.cumsum() / lm_cube.ents.sum(),
                       "% month $": 100 * lm_cube.amt / lm_cube.amt.sum()})
show(f"§8 {LAST_MONTH}: {int(ev_w.tot_ents.iloc[0]):,} active units ({LEVEL_NAME.get(PRIMARY, PRIMARY)})",
     ("Eligibility from the window vs from the month alone", pd.DataFrame(rows)),
     (f"Transactions per unit in {LAST_MONTH}", lm_tab))

## §9 Raw id vs entity

Same candidate rules at each unit level. Entity resolution changes the threshold question materially when large
businesses are split across many raw ids (each id looks like a one-off); the gap between the levels shows how much.

In [ ]:
# §9  Unit levels side by side
rows = []
for r_i, r in enumerate(CANDIDATE_RULES):
    rec = {"rule": cand_name(r_i)}
    for level in LEVELS:
        g = RES[level]["cand"]
        g = g[g.rule_i == r_i][["kept_ents", "tot_ents", "kept_amt", "tot_amt"]].sum()
        rec[f"units kept — {LEVEL_NAME.get(level, level)}"] = g.kept_ents
        rec[f"% kept — {LEVEL_NAME.get(level, level)}"] = 100 * g.kept_ents / g.tot_ents
    rows.append(rec)

# biggest merges among window units at the primary level
prof = spark.read.parquet(P_PROF(PRIMARY))
merges = (prof.where(F.col("n_ids") > 1).orderBy(F.desc("n_ids"))
              .select("name", "name_class", "n_ids", "n_txn", "amt", "n_cust").limit(20).toPandas())
merges["amt"] = merges["amt"].astype(float)
ids_dist = (prof.groupBy(F.least(F.col("n_ids"), F.lit(1000)).alias("n_ids"))
                .agg(F.count("*").alias("units"), F.sum("n_ids").alias("ids")).toPandas().sort_values("n_ids"))
bins = [1, 2, 3, 5, 10, 100, 1000]
ids_dist["bucket"] = [grid_labels(bins)[int(np.searchsorted(bins, v, side="right"))] for v in ids_dist.n_ids]
ids_tab = ids_dist.groupby("bucket", sort=False)[["units", "ids"]].sum().reset_index()
ids_tab["% units"] = 100 * ids_tab.units / ids_tab.units.sum()
ids_tab["% raw ids"] = 100 * ids_tab.ids / ids_tab.ids.sum()

show("§9 Raw id vs entity",
     ("Candidate rules by unit level", pd.DataFrame(rows)),
     (f"Raw ids per unit ({LEVEL_NAME.get(PRIMARY, PRIMARY)}; last bucket capped at 1,000)",
      ids_tab.rename(columns={"bucket": "raw ids per unit"})),
     ("Largest merges in the window", merges.rename(columns={"amt": "$ amount", "n_ids": "ids", "n_txn": "txn",
                                                              "n_cust": "PNC cust"})))

## §10 Customer-side coverage

For each PNC customer: the share of its external (counterparty) dollars and relationships that fall on units kept by
each candidate rule. This is what a customer-facing view built on counterparty metrics (hub exposure, counterparty
quality, TPPP signals) would see. Reported for TM customers and for all customers.

In [ ]:
# §10  Customer coverage per candidate rule
if RUN_CUST_COVERAGE:
    prof = spark.read.parquet(P_PROF(PRIMARY))
    flags = prof.select("unit_id", *[rule_col(r).alias(f"k{i}") for i, r in enumerate(CANDIDATE_RULES)])
    rel = (spark.read.parquet(P_PAIRS)
                .groupBy(F.col(PRIMARY).alias("unit_id"), "cust")
                .agg(F.sum(F.col("amt").cast("double")).alias("amt"), F.max(F.col("cust_is_tm").cast("int")).alias("tm"))
                .join(flags, "unit_id", "inner"))
    per_cust = rel.groupBy("cust").agg(
        F.max("tm").alias("tm"), F.sum("amt").alias("amt"), F.count("*").alias("rels"),
        *[F.sum(F.when(F.col(f"k{i}"), F.col("amt")).otherwise(0.0)).alias(f"ka{i}") for i in range(len(CANDIDATE_RULES))],
        *[F.sum(F.col(f"k{i}").cast("int")).alias(f"kr{i}") for i in range(len(CANDIDATE_RULES))])
    aggs = [F.count("*").alias("custs"), F.sum("amt").alias("amt"), F.sum("rels").alias("rels")]
    for i in range(len(CANDIDATE_RULES)):
        cov = F.when(F.col("amt") > 0, F.col(f"ka{i}") / F.col("amt"))
        rcov = F.col(f"kr{i}") / F.col("rels")
        aggs += [F.sum(f"ka{i}").alias(f"ka{i}"), F.sum(f"kr{i}").alias(f"kr{i}"),
                 F.avg((cov >= 0.99).cast("double")).alias(f"c99_{i}"),
                 F.avg((cov >= 0.90).cast("double")).alias(f"c90_{i}"),
                 F.avg((rcov >= 0.90).cast("double")).alias(f"r90_{i}"),
                 F.avg((F.col(f"kr{i}") == 0).cast("double")).alias(f"none_{i}")]
    cs = per_cust.rollup("tm").agg(*aggs).toPandas()          # tm = 1, 0, and null for all customers
    cs["key"] = cs["tm"].map({1: "TM customers", 0: "non-TM customers"}).fillna("All customers")
    cs = cs.set_index("key")

    def cov_table(key, label):
        row = cs.loc[key]
        out = []
        for i, r in enumerate(CANDIDATE_RULES):
            out.append({"rule": cand_name(i),
                        "% $ covered": 100 * row[f"ka{i}"] / row["amt"],
                        "% relationships covered": 100 * row[f"kr{i}"] / row["rels"],
                        "% customers ≥99% $ covered": 100 * row[f"c99_{i}"],
                        "% customers ≥90% $ covered": 100 * row[f"c90_{i}"],
                        "% customers ≥90% relationships covered": 100 * row[f"r90_{i}"],
                        "% customers with no kept counterparty": 100 * row[f"none_{i}"]})
        return (f"{label}: {int(row['custs']):,} customers", pd.DataFrame(out))

    show(f"§10 Customer-side coverage — {LEVEL_NAME.get(PRIMARY, PRIMARY)}",
         cov_table("TM customers", "TM customers") if "TM customers" in cs.index else None,
         cov_table("All customers", "All customers"))
else:
    show(note("§10 skipped (RUN_CUST_COVERAGE = False)"))

## §11 Save outputs

In [ ]:
# §11  CSVs and summary.json in OUT_DIR
files = []
for level in LEVELS:
    f1 = os.path.join(OUT_DIR, f"cube_{level}.csv"); RES[level]["cube"].to_csv(f1, index=False)
    f2 = os.path.join(OUT_DIR, f"candidates_{level}.csv"); RES[level]["cand"].to_csv(f2, index=False)
    files += [f1, f2]
f3 = os.path.join(OUT_DIR, "rule_search.csv"); SEARCH.to_csv(f3, index=False)
f4 = os.path.join(OUT_DIR, "chosen_composition.csv"); comp.to_csv(f4, index=False)
files += [f3, f4]
summary = {
    "window": [WIN_START, LAST_MONTH], "window_months": WINDOW_MONTHS, "levels": LEVELS,
    "chosen_rule": CHOSEN_RULE, "keep_floors": KEEP_FLOORS,
    "headline": {l: headline(l) for l in LEVELS},
    "chosen_rule_shares": dict(zip(METRICS, map(float, shares(CHOSEN_RULE)))) if on_grid(CHOSEN_RULE) else None,
    "best_per_family": best_per_family[["family", "rule"] + [f"keep_{m}" for m in METRICS]].to_dict("records"),
    "build_log": LOG,
}
f5 = os.path.join(OUT_DIR, "summary.json")
with open(f5, "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
files.append(f5)
show("§11 Saved", *[note(f) for f in files + [os.path.join(OUT_DIR, f) for f in sorted(os.listdir(OUT_DIR)) if f.endswith(".png")]])

## §12 Cleanup — run once the threshold is decided

In [ ]:
# §12  Delete cached intermediates
CLEANUP = False
if CLEANUP:
    gb = path_gb(TMP_ROOT)
    path_delete(TMP_ROOT)
    show(note(f"Deleted {TMP_ROOT} ({gb:,.1f} GB logical)"))
else:
    show(note(f"Kept {TMP_ROOT} ({path_gb(TMP_ROOT):,.1f} GB logical). Set CLEANUP = True to delete."))